In [0]:
%sql
--## Qual a média de valor total (total_amount) recebido em um mês considerando todos os yellow táxis da frota?

SELECT
    pickup_year AS ano_corrida,
    pickup_month AS mes_corrida,
    ROUND(AVG(total_amount), 2) AS media_valor_pago_corrida
FROM gold.taxi_gold
WHERE taxi_type ='yellow'
GROUP BY pickup_year, pickup_month
ORDER BY pickup_year, pickup_month;




In [0]:
%sql
--## Qual a média de passageiros (passenger_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?
SELECT
    HOUR(pickup_datetime) AS hora_inicial_embarque,
    ROUND(AVG(coalesce(passenger_count,0)), 2) AS media_passageiros_por_corrida
FROM gold.taxi_gold
WHERE pickup_month = 5
GROUP BY HOUR(pickup_datetime)
ORDER BY media_passageiros_por_corrida desc;

In [0]:
%python
## Qual o dia da semana em que a duração média das corridas foi maior no mês de maio, considerando todos os táxis da frota?

df = spark.sql("""
SELECT
    date_format(pickup_datetime, 'EEEE') AS dia_semana,
    dayofweek(pickup_datetime) AS dia_semana_num,
    ROUND(AVG(
        (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60
    )) AS media_duracao_corrida_minutos
FROM gold.taxi_gold
WHERE pickup_datetime IS NOT NULL
  AND dropoff_datetime IS NOT NULL
  AND dropoff_datetime > pickup_datetime
  AND pickup_month = 5
GROUP BY
    date_format(pickup_datetime, 'EEEE'),
    dayofweek(pickup_datetime)
""")

## convert to pandas

pdf = df.toPandas()

# Ordena primeiro pelo dia da semana (Domingo → Sábado)
# e depois pela maior duração média

pdf = pdf.sort_values(
    by=["dia_semana_num", "media_duracao_corrida_minutos"],
    ascending=[True, False]
)

## criar grafico
import matplotlib.pyplot as plt

plt.figure()

plt.bar(
    pdf["dia_semana"],
    pdf["media_duracao_corrida_minutos"]
)

plt.xlabel("Dia da semana")
plt.ylabel("Duração média (minutos)")
plt.title("Duração média das corridas por dia da semana - Maio")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()